[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beer-sakthai/openenv-rl-training/blob/main/sakthai-agentic-eval-train/sakthai_grpo_colab.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/beer-sakthai/openenv-rl-training/blob/main/sakthai-agentic-eval-train/sakthai_grpo_colab.ipynb)

> **This notebook can't run on GitHub** (GitHub only shows a static preview). Click a badge above to open it in Colab or Kaggle, then set a GPU runtime and run.

# SakThai GRPO training — Colab / Kaggle (free GPU)

Runs the SakThai agentic-tool-use improvement pipeline on a **free sustained GPU** (Colab T4 or Kaggle 2xT4) — the thing HF Jobs (payment-blocked) and ZeroGPU (bursty, no sustained training) can't do.

Pipeline: base -> **GRPO** (the part that needs sustained GPU) -> push -> eval. Bug fixes from the HF Jobs work are baked in: in-process HermesToolEnvironment (no Docker), use_vllm=False (runs on a T4), 4-bit QLoRA for 7B, conversational prompt format + full tool docstrings, merge-out to bf16 standalone push.

Key finding: **7B is the viable GRPO target** (solves 3/6 tasks -> reward signal). 0.5B/1.5B score 0/6 -> zero reward variance -> GRPO is a no-op on them.

**Runtime -> Change runtime type -> GPU** first. On Kaggle, enable GPU T4 x2.

In [ ]:
# 1. Install deps (trl>=0.29 has environment_factory; jmespath is a hidden hard dep)
!pip install -q "trl>=0.29.0" "transformers>=5.2.0" peft accelerate datasets "openenv==0.4.1" jmespath bitsandbytes huggingface_hub
print("deps installed")

In [ ]:
# 2. Log in. Colab: paste a WRITE token. Kaggle: add HF_TOKEN as a Secret (picked up automatically).
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using Kaggle secret HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 3. Config
BASE_MODEL   = "Nanthasit/sakthai-context-7b-tools"   # 7B is the viable target; 0.5B/1.5B have no GRPO signal
PUSH_TO      = "Nanthasit/sakthai-context-7b-tools-grpo"
USE_QLORA    = True     # True for 7B on a 16GB T4; False (bf16) for 0.5B/1.5B
MAX_STEPS    = 150      # 40 was too short to improve; raise if reward still climbing
EPISODES     = 8
MAX_COMPLETION = 1024
LORA_R       = 16
print(BASE_MODEL, '->', PUSH_TO, '| QLoRA', USE_QLORA, '| steps', MAX_STEPS)

In [ ]:
# 4. Fetch env + set up in-process env wrapper + reward (bypasses broken Docker/client path).
import sys, json
from pathlib import Path
from huggingface_hub import snapshot_download

env_dir = Path(snapshot_download("Nanthasit/hermes-tool-use-rl-env", repo_type="dataset"))
sys.path.insert(0, str(env_dir)); sys.path.insert(0, str(env_dir / "server"))
from models import HermesToolAction
from tasks import TASKS, TASKS_BY_ID
from hermes_tool_env import HermesToolEnvironment

class HermesToolTaskEnvLocal:
    """In-process EnvironmentFactory for trl GRPO. Tool-method docstrings MUST have
    Args: blocks or trl's schema generation raises DocstringParsingException."""
    def __init__(self):
        self._env = HermesToolEnvironment(); self.reward = 0.0; self.done = False
    def reset(self, **kwargs):
        self._env = HermesToolEnvironment(); self._env.reset(); self.reward = 0.0; self.done = False
        tid = kwargs.get('task')
        tid = tid if (tid and tid in TASKS_BY_ID) else sorted(TASKS_BY_ID)[0]
        return self._env.step(HermesToolAction(tool='select_task', task_id=tid)).result
    def terminal(self, command: str) -> str:
        """Run a shell command in the task workspace and return its output.

        Args:
            command: Shell command to execute.
        """
        return self._step(HermesToolAction(tool='terminal', command=command))
    def read_file(self, path: str) -> str:
        """Read a file from the task workspace.

        Args:
            path: Path relative to the workspace root.
        """
        return self._step(HermesToolAction(tool='read_file', path=path))
    def write_file(self, path: str, content: str) -> str:
        """Write (overwrite) a file in the task workspace.

        Args:
            path: Path relative to the workspace root.
            content: Full contents to write.
        """
        return self._step(HermesToolAction(tool='write_file', path=path, content=content))
    def patch(self, path: str, old_string: str, new_string: str) -> str:
        """Find-and-replace a unique substring in a file.

        Args:
            path: Path relative to the workspace root.
            old_string: Exact text to find; must appear exactly once.
            new_string: Replacement text.
        """
        return self._step(HermesToolAction(tool='patch', path=path, old_string=old_string, new_string=new_string))
    def submit(self) -> str:
        """End the episode and grade the task.

        Returns:
            Whether the task passed grading.
        """
        return self._step(HermesToolAction(tool='submit'))
    def _step(self, action):
        if self.done: return 'Episode already ended.'
        obs = self._env.step(action); self.reward = float(obs.reward or 0.0); self.done = bool(obs.done)
        return obs.result

def reward_func(environments, **kwargs):
    return [e.reward for e in environments]

from datasets import Dataset
def build_dataset(episodes):
    prompts, tasks = [], []
    for t in TASKS:
        for _ in range(episodes):
            prompts.append([{ 'role':'user','content':t.prompt }]); tasks.append(t.task_id)
    return Dataset.from_dict({'prompt': prompts, 'task': tasks})
print('env + reward + dataset ready')

In [ ]:
# 5. Resolve base: bare adapter -> merge into base -> local full-model dir (GRPO needs a full model).
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download

def peft_base(repo):
    try: cfg = hf_hub_download(repo, 'adapter_config.json')
    except Exception: return None
    return json.load(open(cfg)).get('base_model_name_or_path')

base_id = peft_base(BASE_MODEL)
if base_id:
    print(f'merging adapter {BASE_MODEL} into {base_id} ...')
    from peft import PeftModel
    b = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.bfloat16)
    PeftModel.from_pretrained(b, BASE_MODEL).merge_and_unload().save_pretrained('./merged_base')
    try: AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained('./merged_base')
    except Exception: AutoTokenizer.from_pretrained(base_id).save_pretrained('./merged_base')
    del b; torch.cuda.empty_cache(); MODEL_PATH = './merged_base'
else:
    MODEL_PATH = BASE_MODEL
print('train from:', MODEL_PATH)

In [ ]:
# 6. GRPO train (use_vllm=False so it runs on a T4; QLoRA to fit 7B). Sustained-GPU step.
import time, torch
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

model_kwargs = {}
if USE_QLORA:
    from transformers import BitsAndBytesConfig
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_R*2,
    target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type='CAUSAL_LM')

args = GRPOConfig(output_dir='./grpo-out', use_vllm=False, max_completion_length=MAX_COMPLETION,
    num_generations=4, per_device_train_batch_size=4, gradient_accumulation_steps=2,
    max_steps=MAX_STEPS, chat_template_kwargs={'enable_thinking': False},
    log_completions=False, bf16=True, report_to=[])

trainer = GRPOTrainer(model=MODEL_PATH, train_dataset=build_dataset(EPISODES),
    reward_funcs=reward_func, args=args, peft_config=peft_config,
    environment_factory=HermesToolTaskEnvLocal, model_init_kwargs=model_kwargs or None)

t0=time.time(); trainer.train(); print(f'GRPO done in {time.time()-t0:.0f}s')
# watch rewards/reward_func/mean: >0 and climbing = learning; flat 0 = no signal (wrong target model)

In [ ]:
# 7. Merge GRPO LoRA out -> bf16 standalone -> push.
from transformers import AutoTokenizer
from huggingface_hub import HfApi
m = trainer.model
if hasattr(m, 'merge_and_unload'): m = m.merge_and_unload()
m = m.to(torch.bfloat16); m.save_pretrained('./grpo-final')
AutoTokenizer.from_pretrained(MODEL_PATH).save_pretrained('./grpo-final')
HfApi().create_repo(PUSH_TO, exist_ok=True)
HfApi().upload_folder(folder_path='./grpo-final', repo_id=PUSH_TO,
    commit_message=f'GRPO from {BASE_MODEL}, {MAX_STEPS} steps (Colab/Kaggle)')
print('pushed ->', PUSH_TO)

In [ ]:
# 8. Quick agentic eval (6 tasks, native template) — did it beat the 3/6 baseline?
import re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
SYS = 'You are a coding agent working in a sandboxed workspace. Inspect the files, make the change the task asks for, then call submit. Call exactly one tool per turn.'
TOOLS = [{'type':'function','function':{'name':n,'description':d,'parameters':{'type':'object','properties':p,'required':r}}} for n,d,p,r in [
  ('terminal','Run a shell command; returns stdout/stderr.',{'command':{'type':'string'}},['command']),
  ('read_file','Read a file.',{'path':{'type':'string'}},['path']),
  ('write_file','Write/overwrite a file.',{'path':{'type':'string'},'content':{'type':'string'}},['path','content']),
  ('patch','Unique-substring find/replace.',{'path':{'type':'string'},'old_string':{'type':'string'},'new_string':{'type':'string'}},['path','old_string','new_string']),
  ('submit','End and grade the task.',{},[])]]
TC = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
tok = AutoTokenizer.from_pretrained(PUSH_TO)
mdl = AutoModelForCausalLM.from_pretrained(PUSH_TO, torch_dtype=torch.bfloat16, device_map='cuda'); mdl.eval()
def gen(msgs):
    enc = tok.apply_chat_template(msgs, tools=TOOLS, add_generation_prompt=True, return_dict=True, return_tensors='pt').to(mdl.device)
    out = mdl.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc['input_ids'].shape[-1]:], skip_special_tokens=True)
passed = 0
for tid in sorted(TASKS_BY_ID):
    env = HermesToolEnvironment(); env.reset()
    obs = env.step(HermesToolAction(tool='select_task', task_id=tid)); r = 0.0
    msgs = [{'role':'system','content':SYS},{'role':'user','content':obs.result}]
    for _ in range(12):
        c = gen(msgs); msgs.append({'role':'assistant','content':c}); m = TC.search(c)
        if not m: msgs.append({'role':'tool','content':'No <tool_call> found.'}); continue
        try: d = json.loads(m.group(1)); obs = env.step(HermesToolAction(tool=d['name'], **(d.get('arguments') or {})))
        except Exception as e: msgs.append({'role':'tool','content':f'err: {e}'}); continue
        msgs.append({'role':'tool','content':obs.result})
        if obs.done: r = float(obs.reward or 0.0); break
    passed += int(r>=1.0); print(f'{tid}: {"PASS" if r>=1 else "fail"}')
print(f'\n=== {PUSH_TO}: {passed}/6 (baseline 7b-tools was 3/6) ===')